In [1]:
import boto3

# Create an EC2 resource
ec2 = boto3.client('ec2')

# Launch a new instance
instance = ec2.run_instances(
    ImageId='ami-0e53db6fd757e38c7',  # Amazon Linux AMI ID
    InstanceType='t2.micro',
    KeyName='Key_Pair',
    MinCount=1,
    MaxCount=1,
    SecurityGroupIds=['sg-0006b3cf7416c4a30']
)

# Print the instance ID
instance_id = instance['Instances'][0]['InstanceId']
print(f'Launched EC2 instance 1 with ID: {instance_id}')

Launched EC2 instance 1 with ID: i-0d3a921898bf6e3e7


In [2]:
import boto3

# Create an EC2 resource
ec2 = boto3.client('ec2')

# Launch a new instance
instance = ec2.run_instances(
    ImageId='ami-0522ab6e1ddcc7055',  # Ubuntu AMI ID
    InstanceType='t2.micro',
    KeyName='Key_Pair',
    MinCount=2,
    MaxCount=2,
    SecurityGroupIds=['sg-0006b3cf7416c4a30']
)

# Print the instance ID
instance_id1 = instance['Instances'][0]['InstanceId']
print(f'Launched EC2 instance 1 with ID: {instance_id1}')
instance_id2 = instance['Instances'][1]['InstanceId']
print(f'Launched EC2 instance 2 with ID: {instance_id2}')

Launched EC2 instance 1 with ID: i-0c8a134e01598551e
Launched EC2 instance 2 with ID: i-07c6f363894afd1e1


In [15]:
import boto3
ec2 = boto3.client('ec2')
response = ec2.describe_instances(
    Filters=[
        {
            'Name': 'instance-state-name',
            'Values': ['running']
        }
    ]
)

for reservation in response['Reservations']:
    for instance in reservation['Instances']:
        print(instance['InstanceId'])

i-0c8a134e01598551e
i-07c6f363894afd1e1
i-0d3a921898bf6e3e7


In [28]:
for reservation in response['Reservations']:
    for instance in reservation['Instances']:
        instance_id = instance['InstanceId']
        response1 = ec2.describe_instance_status(
        InstanceIds=[
            instance_id,
        ],
        )
        print(instance_id)
        print(response1['InstanceStatuses'][0]['SystemStatus']['Details'][0]['Status'])

i-0c8a134e01598551e
passed
i-07c6f363894afd1e1
passed
i-0d3a921898bf6e3e7
passed


In [43]:
import boto3
import paramiko
import time
ec2 = boto3.client('ec2')

instance_id = 'i-0d3a921898bf6e3e7'

# Wait for the instance to be in the 'running' state
waiter = ec2.get_waiter('instance_running')
waiter.wait(InstanceIds=[instance_id])

# Connect to the instance and install Apache
instance = ec2.describe_instances(InstanceIds=[instance_id])['Reservations'][0]['Instances'][0]
public_ip = instance['PublicIpAddress']
ssh_client = paramiko.SSHClient()
ssh_client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
key = paramiko.RSAKey.from_private_key_file('Key_Pair.pem')
ssh_client.connect(hostname=public_ip, username='ec2-user', pkey=key)
stdin, stdout, stderr = ssh_client.exec_command('sudo yum install -y httpd')
print(stdout.read().decode())
print(stderr.read().decode())
time.sleep(5)  # Wait for 5 seconds
stdin, stdout, stderr = ssh_client.exec_command('sudo systemctl start httpd')
print(stdout.read().decode())
print(stderr.read().decode())

print(f"Apache web server is now running on instance {instance_id} at {public_ip}")

Last metadata expiration check: 1:40:22 ago on Sun Sep  8 16:03:59 2024.
Package httpd-2.4.62-1.amzn2023.x86_64 is already installed.
Dependencies resolved.
Nothing to do.
Complete!




Apache web server is now running on instance i-0d3a921898bf6e3e7 at 43.205.196.126


In [44]:
response = ec2.describe_instances(
    Filters=[
        {
            'Name': 'instance-state-name',
            'Values': ['running']
        }
    ]
)

instance_ids = [instance['InstanceId'] for reservation in response['Reservations'] for instance in reservation['Instances']]
ec2.stop_instances(InstanceIds=instance_ids)
print(f"Stopped the following instances: {', '.join(instance_ids)}")

Stopped the following instances: i-0c8a134e01598551e, i-07c6f363894afd1e1, i-0d3a921898bf6e3e7


In [45]:
response = ec2.describe_instances(
    Filters=[
        {
            'Name': 'instance-state-name',
            'Values': ['running']
        }
    ]
)

for reservation in response['Reservations']:
    for instance in reservation['Instances']:
        instance_id = instance['InstanceId']
        print(f"Terminating instance: {instance_id}")
        ec2.terminate_instances(InstanceIds=[instance_id])

Terminating instance: i-0c8a134e01598551e
Terminating instance: i-07c6f363894afd1e1
Terminating instance: i-0d3a921898bf6e3e7
